[Chapter 6] Feature Engineering for Time Series Forecasting

In [2]:
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings("ignore")

In [3]:
# read train test val split data (with missing values imputed)

train_df = pd.read_parquet("./data/london_smart_meters/preprocessed/selected_blocks_train_missing_imputed.parquet")
val_df = pd.read_parquet("./data/london_smart_meters/preprocessed/selected_blocks_val_missing_imputed.parquet")
test_df = pd.read_parquet("./data/london_smart_meters/preprocessed/selected_blocks_test_missing_imputed.parquet")

# add train, validation, and test tags to distinguish then before combining

train_df['type'] = 'train'
val_df['type'] = 'val'
test_df['type'] = 'test'
full_df = pd.concat([train_df, val_df, test_df]).sort_values(['LCLid','timestamp'])
del train_df, test_df, val_df

time delay embedding - lags or backshift

In [4]:
from src.feature_engineering.autoregressive_features import add_lags

# create a list of integers, denoting all the lags we need to create as features
lags = (
    (np.arange(5) + 1).tolist()
    + (np.arange(5) + 46).tolist()
    + (np.arange(5) + (48 + 7) - 2).tolist()
)
full_df, added_features = add_lags(
    full_df, lags = lags, column='energy_consumption', ts_id='LCLid', use_32_bit=True
)

time delay embedding - rolling window aggregation

In [5]:
# take past n consecutive observations in the window

from src.feature_engineering.autoregressive_features import add_rolling_features

full_df, added_features = add_rolling_features(
    full_df,
    rolls=[3, 6, 12, 48],           # all the windows over which we need to calculate the agg statistics
    column='energy_consumption',    # the name of the column to be lagged
    agg_funcs=['mean','std'],       
    ts_id='LCLid',                  # unique ts id
    use_32_bit=True,                
)

time delay embedding - seasonal rolling window aggregation

In [6]:
# take seasonal window (e.g. mean of y_{t-M}, y_{t-2M}, y_{t-3M})

from src.feature_engineering.autoregressive_features import add_seasonal_rolling_features

full_df, added_features = add_seasonal_rolling_features(
    full_df,
    rolls=[3],
    seasonal_periods=[48, 48*7],    # list of periods that should be used in the seasonal rolling windows
    column = "energy_consumption",  
    agg_funcs=["mean", "std"],
    ts_id = 'LCLid',
    use_32_bit=True
)

exponentially weighted moving average (ewma)

In [7]:
# ewma_{T} = alpha x y_{T} + (1-alpha) x ewma_{T-1}
# weight of each term: w_{T-K} = alpha x (1-alpha)^k
# alpha = 2 / (1+ span) span is the number of periods at which the decayed weights approach zero
 
from src.feature_engineering.autoregressive_features import add_ewma

full_df, added_features = add_ewma(
    full_df,
    spans=[48 * 60, 48 * 7, 48],
    column = 'energy_consumption',
    ts_id='LCLid',
    use_32_bit=True,
)

temporal embedding - calendar feature

In [ ]:
# capture the passage of time as categoircal variables in ML model

from src.feature_engineering.temporal_features import add_temporal_features

full_df, added_features = add_temporal_features(
    full_df,
    field_name='timestamp', # the column containing datetime that should be used to create features
    frequency='30min',      
    add_elapsed=True,       # turn the creation of the time elapsed feature on
    drop=False,
    use_32_bit=True,
)

temporal embedding - fourier term

In [ ]:
# continuous representation of seasonality

from src.feature_engineering.temporal_features import bulk_add_fourier_features

full_df, added_features = bulk_add_fourier_features(
    full_df,
    columns_to_encode=['timestamp_Month', 'timestamp_Hour', 'timestamp_Minute'],    # list of calendar features using fourier terms
    max_values=[12, 24, 60],
    n_fourier_terms=5,
    use_32_bit=True,
)